## Initialization for Local Development

In [1]:
import kagglehub
from confluent_kafka.admin import AdminClient, NewTopic
from pyspark.sql import SparkSession
spark = (SparkSession.builder.appName("MyApp")
.config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.1,org.apache.kafka:kafka-clients:3.6.0,org.apache.spark:spark-streaming-kafka-0-10_2.13:4.2.1")
.master("local[*]")
.getOrCreate())

/Users/minhman/Downloads/Workspace/Big Data/Lab-1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/29 17:02:40 WARN Utils: Your hostname, Khng-phi-Minh-Mn.local, resolves to a loopback address: 127.0.0.1; using 10.229.189.227 instead (on interface en0)
26/10/29 17:02:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/minhman/Downloads/Workspace/Big%20Data/Lab-1/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/minhman/.ivy2.5.2/cache
The jars for the packages stored in: /Users/minhman/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a depen

## Download Kaggle dataset

In [ ]:
path = kagglehub.dataset_download("grouplens/movielens-latest-small")
df_ratings = spark.read.csv(path + "/ratings.csv", header=True, inferSchema=True)
df_movies = spark.read.csv(path + "/movies.csv", header=True, inferSchema=True)
df_tags = spark.read.csv(path + "/tags.csv", header=True, inferSchema=True)
df_ratings.show(1)
df_movies.show(1)
df_tags.show(1)

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
+------+-------+------+---------+
only showing top 1 row
+-------+----------------+--------------------+
|movieId|           title|              genres|
+-------+----------------+--------------------+
|      1|Toy Story (1995)|Adventure|Animati...|
+-------+----------------+--------------------+
only showing top 1 row
+------+-------+-----+----------+
|userId|movieId|  tag| timestamp|
+------+-------+-----+----------+
|     2|  60756|funny|1445714994|
+------+-------+-----+----------+
only showing top 1 row


26/10/29 17:09:23 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 290716 ms exceeds timeout 120000 ms
26/10/29 17:09:24 WARN SparkContext: Killing executors is not supported by current scheduler.
26/10/29 17:09:24 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:34)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.stora

## Create topics

In [3]:
# First, delete topics if they exist
admin_client = AdminClient({'bootstrap.servers': 'localhost:9092'})
admin_client.delete_topics(['ratings', 'movies', 'tags'], operation_timeout=10)
topic_metadata = admin_client.list_topics(timeout=10)
print("Deleted if exists")
# Create new topics
new_topics = [NewTopic(topic="ratings", num_partitions=1, replication_factor=3),
              NewTopic(topic="movies", num_partitions=1, replication_factor=3),
              NewTopic(topic="tags", num_partitions=1, replication_factor=3)]
fs = admin_client.create_topics(new_topics)
for topic, f in fs.items():
    try:
        f.result() # The result itself is None
        print(f"Topic {topic} created")
    except Exception as e:
        print(f"Failed to create topic {topic}: {e}")

Deleted if exists
Topic ratings created
Topic movies created
Topic tags created


## Push data to Kafka using Spark

In [4]:
# Push the dataframes to kafka broker
df_ratings.selectExpr("to_json(struct(*)) AS value") \
    .write \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("topic", "ratings") \
    .save()
df_movies.selectExpr("to_json(struct(*)) AS value") \
    .write \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("topic", "movies") \
    .save()
df_tags.selectExpr("to_json(struct(*)) AS value") \
    .write \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("topic", "tags") \
    .save()
# Check if data is in Kafka topics, however it's stored in binary format, and in "value" column
df_ratings_kafka = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "ratings") \
    .load()

df_ratings_kafka = df_ratings_kafka.selectExpr("CAST(value AS STRING)")
df_ratings_kafka.show(1)

+--------------------+
|               value|
+--------------------+
|{"userId":1,"movi...|
+--------------------+
only showing top 1 row


## Exercises

### Task 1: Get data from Kafka from 3 brokers

In [5]:
from pyspark.sql.functions import col, from_json, avg, count, round, desc, asc, countDistinct, trim
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType, LongType

KAFKA_SINGLE_BROKER = "localhost:9092"
KAFKA_BROKERS = "localhost:9092,localhost:9192,localhost:9292"

# 1. Define explicit schemas for each topic
ratings_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("rating", DoubleType(), True),
    StructField("timestamp", LongType(), True)
])

movies_schema = StructType([
    StructField("movieId", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("genres", StringType(), True)
])

tags_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("tag", StringType(), True),
    StructField("timestamp", LongType(), True)
])

# 2. Helper function to read from Kafka and parse JSON binary data
def read_kafka_topic(topic_name, schema):
    raw_df = (spark.read
        .format("kafka")
        .option("kafka.bootstrap.servers", KAFKA_SINGLE_BROKER)
        .option("subscribe", topic_name)
        .option("startingOffsets", "earliest")
        .load())
    
    parsed_df = (raw_df
        .selectExpr("CAST(value AS STRING) AS json_str")
        .select(from_json(col("json_str"), schema).alias("data"))
        .select("data.*")
        .filter(col("movieId").isNotNull()) # Filter corrupt or null rows
    )
    return parsed_df
    # return raw_df.selectExpr("CAST(value AS STRING) AS json_str")

# Read and parse all 3 topics
kafka_ratings_df = read_kafka_topic("ratings", ratings_schema)
kafka_movies_df = read_kafka_topic("movies", movies_schema)
kafka_tags_df = read_kafka_topic("tags", tags_schema)

print("Ratings DataFrame from Kafka:")
kafka_ratings_df.show(5)

print("Movies DataFrame from Kafka:")
kafka_movies_df.show(5)

print("Tags DataFrame from Kafka:")
kafka_tags_df.show(5)


Ratings DataFrame from Kafka:
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982224|
|     1|     47|   5.0|964983815|
|     1|     50|   5.0|964982931|
+------+-------+------+---------+
only showing top 5 rows
Movies DataFrame from Kafka:
+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|
+-------+--------------------+--------------------+
only showing top 5 rows
Tags DataFrame from Kafka:
+------+-------+---------------+----------+
|userId|movieId|            tag| timestamp|
+---

### Task 2: Top 5 Movies by Average Rating (Rating Count > 30)

In [9]:
# Task 2: Top 5 movies (by Id) by average rating, whose rating count is higher than 30
top_5_movies = (kafka_ratings_df
    .groupBy("movieId")
    .agg(
        avg("rating").alias("avg_rating"),
        count("rating").alias("rating_count")
    )
    .filter(col("rating_count") > 30)
    .join(kafka_movies_df, on="movieId", how="inner")
    .select("movieId", "title", "avg_rating", "rating_count")
    .orderBy(desc("avg_rating"), desc("rating_count"), asc("movieId"))
    .limit(5)
)

print("Top 5 movies by average rating (rating count > 30):")
top_5_movies.show(truncate=False)


Top 5 movies by average rating (rating count > 30):


+-------+--------------------------------+-----------------+------------+
|movieId|title                           |avg_rating       |rating_count|
+-------+--------------------------------+-----------------+------------+
|318    |Shawshank Redemption, The (1994)|4.429022082018927|317         |
|1204   |Lawrence of Arabia (1962)       |4.3              |45          |
|858    |Godfather, The (1972)           |4.2890625        |192         |
|2959   |Fight Club (1999)               |4.272935779816514|218         |
|1276   |Cool Hand Luke (1967)           |4.271929824561403|57          |
+-------+--------------------------------+-----------------+------------+



### Task 3: 5 Worst Tags (Lowest Average Rating)

In [10]:
# Task 3: Get 5 worst tags (associated with lowest average rating)
# Compute each movie average rating first
movie_avg_df = (kafka_ratings_df
    .groupBy("movieId")
    .agg(avg("rating").alias("movie_avg_rating"))
)

# Join tags with movie average ratings
tags_ratings = (kafka_tags_df
    .select("movieId", trim(col("tag")).alias("tag"))
    .filter(col("tag").isNotNull() & (col("tag") != ""))
    .join(movie_avg_df, on="movieId", how="inner")
)

# Compute average rating per tag and find 5 worst
worst_5_tags_df = (tags_ratings
    .groupBy("tag")
    .agg(
        avg("movie_avg_rating").alias("avg_rating"),
        count("movieId").alias("tag_count")
    )
    .orderBy(asc("avg_rating"), desc("tag_count"))
    .limit(5)
)

print("5 Worst Tags:")
worst_5_tags_df.show(truncate=False)

# Extract the list of worst tags for Task 4
worst_tags_list = [row["tag"] for row in worst_5_tags_df.collect()]
print("List of 5 worst tags:", worst_tags_list)


5 Worst Tags:


+--------+------------------+---------+
|tag     |avg_rating        |tag_count|
+--------+------------------+---------+
|symbolic|0.5               |1        |
|shark   |1.4166666666666667|1        |
|stage   |1.75              |1        |
|Tokyo   |2.0               |1        |
|SNL     |2.1               |1        |
+--------+------------------+---------+



List of 5 worst tags: ['symbolic', 'shark', 'stage', 'Tokyo', 'SNL']


### Task 4: For each movie associated with these tags, check their average ratings and how many movies have these tags.


In [11]:
# 4.1 Check how many movies have these worst tags
movies_count_per_tag = (kafka_tags_df
    .select("movieId", trim(col("tag")).alias("tag"))
    .filter(col("tag").isin(worst_tags_list))
    .groupBy("tag")
    .agg(countDistinct("movieId").alias("distinct_movie_count"))
    .orderBy("distinct_movie_count")
)

print("Number of distinct movies having each worst tag:")
movies_count_per_tag.show(truncate=False)

# 4.2 Check average ratings for each movie associated with these tags
movies_associated = (kafka_tags_df
    .select("movieId", trim(col("tag")).alias("tag"))
    .filter(col("tag").isin(worst_tags_list))
    .distinct()
    .join(kafka_movies_df, on="movieId", how="inner")
    .join(movie_avg_df, on="movieId", how="inner")
    .select("tag", "movieId", "title", col("movie_avg_rating").alias("movie_avg_rating"))
    .orderBy("tag", "movie_avg_rating")
)

print("Each movie associated with these worst tags and their average rating:")
movies_associated.show(50, truncate=False)


Number of distinct movies having each worst tag:


+--------+--------------------+
|tag     |distinct_movie_count|
+--------+--------------------+
|Tokyo   |1                   |
|shark   |1                   |
|SNL     |1                   |
|symbolic|1                   |
|stage   |1                   |
+--------+--------------------+

Each movie associated with these worst tags and their average rating:


+--------+-------+------------------------------+------------------+
|tag     |movieId|title                         |movie_avg_rating  |
+--------+-------+------------------------------+------------------+
|SNL     |2296   |Night at the Roxbury, A (1998)|2.1               |
|Tokyo   |6407   |Walk, Don't Run (1966)        |2.0               |
|shark   |1389   |Jaws 3-D (1983)               |1.4166666666666667|
|stage   |8943   |Being Julia (2004)            |1.75              |
|symbolic|26717  |Begotten (1990)               |0.5               |
+--------+-------+------------------------------+------------------+

